<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/conversor_bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook para união de bases - Painel

>Trata-se de notebook para extração de dados porovenientes das eleições do TSE dos anos de 2018 e 2022.

**Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

## Etapa 1: Leitura das duas bases


#### **IMPORTANTE**: Há dois arquivos com extensão .csv: um com **dimensão eleitoral** em que consta as 3 variáveis 'austeridade', 'mesmopartido' e 'reeleito' e outro com a **dimensão fiscal**,  com dados da gestão do mandato do eleito.



1. Dimensão eleitoral: eleicoes_uf_2018_2022_austeridade_reeleito_mesmopartido.csv

2. Dimensão fiscal: fiscal_uf_2019_2024.csv

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df_eleitoral = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/2026-mestrado-pep/refs/heads/main/data/eleitoral_uf_2018_2022_austeridade_reeleito_mesmopartido.csv')

df_fiscal = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/2026-mestrado-pep/refs/heads/main/data/fiscal_uf_2019_2024.csv')


# Por default, eu já crio as colunas vazias no dataset fiscal.
df_fiscal['austeridade'] = np.nan
df_fiscal['mesmopartido'] = np.nan
df_fiscal['reeleito'] = np.nan



In [6]:
df_eleitoral.head(10)

,SG_UF,ANO_ELEICAO,DS_CARGO,NM_CANDIDATO,SG_PARTIDO,austeridade,reeleito,mesmopartido
0,AC,2018,GOVERNADOR,GLADSON DE LIMA CAMELI,PP,0,0,0
1,AC,2022,GOVERNADOR,GLADSON DE LIMA CAMELI,PP,0,1,1
2,AL,2018,GOVERNADOR,JOSE RENAN VASCONCELOS CALHEIROS FILHO,MDB,0,1,1
3,AL,2022,GOVERNADOR,PAULO SURUAGY DO AMARAL DANTAS,MDB,0,0,1
4,AM,2018,GOVERNADOR,WILSON MIRANDA LIMA,PSC,0,0,0
5,AM,2022,GOVERNADOR,WILSON MIRANDA LIMA,UNIÃO,1,1,0
6,AP,2018,GOVERNADOR,ANTONIO WALDEZ GÓES DA SILVA,PDT,0,0,1
7,AP,2022,GOVERNADOR,CLÉCIO LUÍS VILHENA VIEIRA,SOLIDARIEDADE,0,0,0
8,BA,2018,GOVERNADOR,RUI COSTA DOS SANTOS,PT,0,1,1
9,BA,2022,GOVERNADOR,JERÔNIMO RODRIGUES SOUZA,PT,0,0,1


In [7]:
df_fiscal.head(10)

,uf,ano,dcl,pop,pib_prc,rcl,rcla,dtp,percentual_dtp_rcla,austeridade,mesmopartido,reeleito
0,AC,2019,3.116892e+09,869265,1.563002e+10,5.357456e+09,5.357456e+09,2.878921e+09,53.74,NaN,NaN,NaN
1,AC,2020,3.337030e+09,881935,1.647637e+10,5.702871e+09,5.702871e+09,3.004991e+09,52.69,NaN,NaN,NaN
2,AC,2021,2.847799e+09,894470,2.137444e+10,6.690646e+09,6.651120e+09,3.421543e+09,51.44,NaN,NaN,NaN
3,AC,2022,2.505322e+09,906876,2.367614e+10,7.994707e+09,7.968319e+09,3.694040e+09,46.36,NaN,NaN,NaN
4,AC,2023,2.051524e+09,906876,2.629132e+10,8.573004e+09,8.496046e+09,4.113347e+09,48.41,NaN,NaN,NaN
5,AC,2024,1.978575e+09,829780,0.000000e+00,1.011123e+10,1.000246e+10,4.678325e+09,46.77,NaN,NaN,NaN
6,AL,2019,6.404122e+09,3322820,5.896373e+10,8.559007e+09,8.559007e+09,3.826568e+09,44.71,NaN,NaN,NaN
7,AL,2020,5.813490e+09,3337357,6.320235e+10,1.005950e+10,1.004934e+10,3.997128e+09,39.78,NaN,NaN,NaN
8,AL,2021,4.758264e+09,3351543,7.626562e+10,1.252891e+10,1.246151e+10,4.436804e+09,35.60,NaN,NaN,NaN
9,AL,2022,7.245315e+09,3365351,7.606581e+10,1.317791e+10,1.314893e+10,5.378105e+09,40.90,NaN,NaN,NaN


## Etapa 2: Geração da variável austeridade a partir da dimensão eleitoral

Verifica se o candidato vencedor é do agremiação partidária que defende austeridade das contas a partir da dimensão eleitoral. Na sequência, apenas constata-se se o resultado do pleito 2022 foi copiado corretamente nos anos fiscais de 2023 e 2024, por exemplo.



In [8]:
import numpy as np
lista_anos = list(df_fiscal.ano.unique())
lista_uf = list(df_fiscal.uf.unique())

lista_UF_2018_austeridade_eleito = df_eleitoral[ (df_eleitoral.austeridade == 1) & (df_eleitoral.ANO_ELEICAO == 2018)]['SG_UF'].unique()
lista_UF_2022_austeridade_eleito = df_eleitoral[ (df_eleitoral.austeridade == 1) & (df_eleitoral.ANO_ELEICAO == 2022)]['SG_UF'].unique()

grupo_austeridade_eleito = {
    2018: lista_UF_2018_austeridade_eleito,
    2022: lista_UF_2022_austeridade_eleito
}


# Para quaisquer casos, todos zerados no inicio
df_fiscal['austeridade'] = 0

# Para cada UF e ano , verificar se o ano de mandato corresponde ao pleito de 2018 ou 2022 e se austero
for uf in lista_uf:
  for ano in lista_anos:
      esta_grupo_austeridade = (( ((2018 < ano <= 2022) and (uf in grupo_austeridade_eleito[2018]) ) or ((ano > 2022)  and (uf in grupo_austeridade_eleito[2022])) ))
      if esta_grupo_austeridade:
          df_fiscal.loc[ (df_fiscal['uf'] == uf) & (df_fiscal['ano'] == ano), 'austeridade'] = int(esta_grupo_austeridade)




df_fiscal.head(50)

,uf,ano,dcl,pop,pib_prc,rcl,rcla,dtp,percentual_dtp_rcla,austeridade,mesmopartido,reeleito
0,AC,2019,3.116892e+09,869265,1.563002e+10,5.357456e+09,5.357456e+09,2.878921e+09,53.74,0,NaN,NaN
1,AC,2020,3.337030e+09,881935,1.647637e+10,5.702871e+09,5.702871e+09,3.004991e+09,52.69,0,NaN,NaN
2,AC,2021,2.847799e+09,894470,2.137444e+10,6.690646e+09,6.651120e+09,3.421543e+09,51.44,0,NaN,NaN
3,AC,2022,2.505322e+09,906876,2.367614e+10,7.994707e+09,7.968319e+09,3.694040e+09,46.36,0,NaN,NaN
4,AC,2023,2.051524e+09,906876,2.629132e+10,8.573004e+09,8.496046e+09,4.113347e+09,48.41,0,NaN,NaN
5,AC,2024,1.978575e+09,829780,0.000000e+00,1.011123e+10,1.000246e+10,4.678325e+09,46.77,0,NaN,NaN
6,AL,2019,6.404122e+09,3322820,5.896373e+10,8.559007e+09,8.559007e+09,3.826568e+09,44.71,0,NaN,NaN
7,AL,2020,5.813490e+09,3337357,6.320235e+10,1.005950e+10,1.004934e+10,3.997128e+09,39.78,0,NaN,NaN
8,AL,2021,4.758264e+09,3351543,7.626562e+10,1.252891e+10,1.246151e+10,4.436804e+09,35.60,0,NaN,NaN
9,AL,2022,7.245315e+09,3365351,7.606581e+10,1.317791e+10,1.314893e+10,5.378105e+09,40.90,0,NaN,NaN


In [9]:
df_eleitoral[(df_eleitoral['ANO_ELEICAO']==2022)]['austeridade'].value_counts()

,count
austeridade,
1,14
0,13


In [10]:
df_fiscal[(df_fiscal['ano']==2023)]['austeridade'].value_counts()

,count
austeridade,
1,14
0,13


In [11]:
df_fiscal[(df_fiscal['ano']==2024)]['austeridade'].value_counts()

,count
austeridade,
1,14
0,13


---

## Etapa 3: Geração da variável 'mesmopartido'
> Verifica se o candidato vencedor é do **mesmo partido** do governador anterior a partir da dimensão eleitoral. Na sequência, apenas é feita uma checagem  se o resultado do pleito 2022 foi copiado corretamente nos anos fiscais de 2023 e 2024, por exemplo.

In [12]:
grupo_mesmopartido = {
    2018: df_eleitoral[ (df_eleitoral.mesmopartido == 1) & (df_eleitoral.ANO_ELEICAO == 2018)]['SG_UF'].unique(),
    2022: df_eleitoral[ (df_eleitoral.mesmopartido == 1) & (df_eleitoral.ANO_ELEICAO == 2022)]['SG_UF'].unique()
}

# Para quaisquer casos, todos zerados no inicio
df_fiscal['mesmopartido'] = 0

for uf in lista_uf:
  for ano in lista_anos:
      esta_grupo_mesmopartido = (( ((2018 < ano <= 2022) and (uf in grupo_mesmopartido[2018]) ) or ((ano > 2022)  and (uf in grupo_mesmopartido[2022])) ))
      if esta_grupo_mesmopartido:
          df_fiscal.loc[ (df_fiscal['uf'] == uf) & (df_fiscal['ano'] == ano), 'mesmopartido'] = int(esta_grupo_mesmopartido)

In [13]:
df_eleitoral[(df_eleitoral['ANO_ELEICAO']==2022)]['mesmopartido'].value_counts()

,count
mesmopartido,
1,15
0,12


In [14]:
df_fiscal[(df_fiscal['ano']==2023)]['mesmopartido'].value_counts()

,count
mesmopartido,
1,15
0,12


In [15]:
df_fiscal[(df_fiscal['ano']==2024)]['mesmopartido'].value_counts()

,count
mesmopartido,
1,15
0,12


## Etapa 4: Geração da variável 'reeleito'

>Verifica se o candidato vencedor foi reeleito a partir da dimensão eleitoral. Na sequência, apenas é feita uma checagem se o resultado de pleito 2022 foi copiado corretamente nos anos fiscais de 2023 e 2024, por exemplo.


In [16]:
grupo_reeleito = {
    2018: df_eleitoral[ (df_eleitoral.reeleito == 1) & (df_eleitoral.ANO_ELEICAO == 2018)]['SG_UF'].unique(),
    2022: df_eleitoral[ (df_eleitoral.reeleito == 1) & (df_eleitoral.ANO_ELEICAO == 2022)]['SG_UF'].unique()
}

# Para quaisquer casos, todos zerados no inicio
df_fiscal['reeleito'] = 0

# Para cada UF e ano , verificar se o ano de mandato corresponde ao pleito de 2018 ou 2022 e se austero
for uf in lista_uf:
  for ano in lista_anos:
      esta_grupo_reeleito = (( ((2018 < ano <= 2022) and (uf in grupo_reeleito[2018]) ) or ((ano > 2022)  and (uf in grupo_reeleito[2022])) ))
      if esta_grupo_reeleito:
          df_fiscal.loc[ (df_fiscal['uf'] == uf) & (df_fiscal['ano'] == ano), 'reeleito'] = int(esta_grupo_reeleito)



In [17]:
df_eleitoral[(df_eleitoral['ANO_ELEICAO']==2022)]['reeleito'].value_counts()

,count
reeleito,
0,16
1,11


In [18]:
df_fiscal[(df_fiscal['ano']==2023)]['reeleito'].value_counts()

,count
reeleito,
0,16
1,11


In [19]:
df_fiscal[(df_fiscal['ano']==2024)]['reeleito'].value_counts()

,count
reeleito,
0,16
1,11


## Etapa 5: Correspondência e Checagens entre dimensão eleitoral e dimensão fiscal

1. Na dimensão eleitoral há apenas 54 linhas (27 UF e anos 2018 e 2022).

2. Na dimensão fiscal há 162 linhas (27 UF e anos: 2019, 2020, 2021, 2022, 2023 e 2024)

3. Para cada linha das 3 variáveis eleitorais no ano de 2018, 4 linhas correspondentes(2019-2022) na dimensão fiscal terão o mesmo valor.

4. Para cada linha das 3 variáveis fiscais no ano de 2022, 2 linhas correspondentes(2023-2024) na dimensão fiscal terão o mesmo valor.

>4A. Pega os valores das 3 variáveis na dimensão eleitoral em 2018 e 2022.  

>4B. Pega os valores das 3 variáveis na dimensão fiscal no mandato1(2019-2022) compara com a dimensão eleitoral no ano 2018. No mandato2(2023,2024), compara com resultado eleitoral de 2022. Todas 3 variáveis tem que possuir mesmo valor.

In [20]:

#  4A - Pegando os valores da dimensão eleitoral das 3 variáveis em 2018 e 2022
k, v = [], []
for ano in [2018, 2022]:
  for var in ['austeridade', 'mesmopartido', 'reeleito']:
      k.append(f"qtd_{var}_{ano}")
      v.append(df_eleitoral.loc[df_eleitoral["ANO_ELEICAO"] == ano, var].value_counts())


placar_eleitoral = dict(zip(k, v))
lista_vars = ['austeridade', 'reeleito', 'mesmopartido']

# 4B Pega os valores em 2019 até 2024 e compara com o respectivo pleito eleitoral (2018 ou 2022)
for ano in lista_anos:
    for var in lista_vars:
        qtd_fiscal = tuple(df_fiscal.loc[df_fiscal["ano"] == ano, var].value_counts().values)
        y = 2018 if (ano <= 2022) else 2022
        qtd_eleitoral = tuple(placar_eleitoral[f"qtd_{var}_{y}"].values)
        if qtd_eleitoral != qtd_fiscal:
            print(f"Houve divergência de Valores no Ano: {ano} - Variável: {var}\n")
            print(f"Qtd_eleitoral: 0: {qtd_eleitoral[0]}, 1: {qtd_eleitoral[0]} \n")
            print(f"Qtd_fiscal: 0: {qtd_eleitoral[0]}, 1: {qtd_eleitoral[0]} \n")
            break
    else:
        print(f"Não houve divergência de Valores no Ano: {ano}\n")

Não houve divergência de Valores no Ano: 2019

Não houve divergência de Valores no Ano: 2020

Não houve divergência de Valores no Ano: 2021

Não houve divergência de Valores no Ano: 2022

Não houve divergência de Valores no Ano: 2023

Não houve divergência de Valores no Ano: 2024



## Etapa 7 - Criação da váriável Y (dependente)

Gestão_Fiscal_Responsável = 1 se (dtp/rcla < 49%) e 0 , em caso contrário.



In [21]:
df_fiscal.head(10)

,uf,ano,dcl,pop,pib_prc,rcl,rcla,dtp,percentual_dtp_rcla,austeridade,mesmopartido,reeleito
0,AC,2019,3.116892e+09,869265,1.563002e+10,5.357456e+09,5.357456e+09,2.878921e+09,53.74,0,0,0
1,AC,2020,3.337030e+09,881935,1.647637e+10,5.702871e+09,5.702871e+09,3.004991e+09,52.69,0,0,0
2,AC,2021,2.847799e+09,894470,2.137444e+10,6.690646e+09,6.651120e+09,3.421543e+09,51.44,0,0,0
3,AC,2022,2.505322e+09,906876,2.367614e+10,7.994707e+09,7.968319e+09,3.694040e+09,46.36,0,0,0
4,AC,2023,2.051524e+09,906876,2.629132e+10,8.573004e+09,8.496046e+09,4.113347e+09,48.41,0,1,1
5,AC,2024,1.978575e+09,829780,0.000000e+00,1.011123e+10,1.000246e+10,4.678325e+09,46.77,0,1,1
6,AL,2019,6.404122e+09,3322820,5.896373e+10,8.559007e+09,8.559007e+09,3.826568e+09,44.71,0,1,1
7,AL,2020,5.813490e+09,3337357,6.320235e+10,1.005950e+10,1.004934e+10,3.997128e+09,39.78,0,1,1
8,AL,2021,4.758264e+09,3351543,7.626562e+10,1.252891e+10,1.246151e+10,4.436804e+09,35.60,0,1,1
9,AL,2022,7.245315e+09,3365351,7.606581e+10,1.317791e+10,1.314893e+10,5.378105e+09,40.90,0,1,1


In [22]:
df_fiscal['gestao_fiscal_responsavel'] = np.where((df_fiscal.dtp/df_fiscal.rcla)<0.49, 1, 0)

## Etapa Final: Criação do Painel em formato csv

In [23]:
import os

# Criar o diretório 'data' se ele não existir
os.makedirs('./data', exist_ok=True)

df_fiscal.to_csv('./data/painel_uf_2019_2024_completo.csv', index=False)

---